In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Multi-Class XGBoost OOF Probabilities -> Stage 2 Logistic Regression with Controlled Class Count Regulation (`models/xgboost_feng_stage1_extreme.ipynb`)

This notebook implements a **2-Stage Stacked Generalization Pipeline** for 5-class ESI triage using **Controlled Class Count Subsampling (without inverse class weighting)** and **`mlr3tuning` + `bbotk` Hyperparameter Optimization**:

### System Architecture & Key Improvements
1. **Feature Set (13 Clinical Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and 10 vital sign anomaly flags (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
2. **Controlled Majority Subsampling (No Inverse Class Weighting)**:
   - Instead of applying extreme numeric weights (e.g. 105.8x for ESI 1), majority classes (ESI 2, 3, 4) are regulated/capped to balanced target ratios relative to minority classes.
   - Retains 100% of rare ESI 1 and ESI 5 rows while preventing ESI 3 majority domination.
3. **Stage 1 XGBoost Hyperparameter Optimization (`mlr3tuning` + `bbotk`)**:
   - Uses `mlr3`, `mlr3learners`, `mlr3tuning`, `bbotk`, and `paradox` with `rsmp("cv")` to tune Stage 1 XGBoost over multi-class log-loss.
4. **Stage 1 XGBoost 5-Fold Stratified OOF Generation**:
   - Trains 5-class XGBoost models (`objective = "multi:softprob"`, `num_class = 5`, `evals = list(...)`) without weight distortion.
   - Generates Out-Of-Fold (OOF) predicted probability features ($P_1, P_2, P_3, P_4, P_5$) without data leakage.
5. **Stage 2 Unweighted Meta-Learner (Logistic Regression)**:
   - Fits Stage 2 Multinomial Logistic Regressor (`multinom`) on Stage 1 OOF probability features.
6. **Comprehensive Multi-Class Evaluation & Artifact Exports**:
   - Evaluates Validation and Test sets with 5x5 confusion matrices, class count comparison tables across all 5 ESI levels, Accuracy, Precision, Recall, F1, PR-AUC, and ROC-AUC.
   - Exports `plots/xgboost_stage1_lr_stage2_metrics.png`, `reports/xgboost_stage1_lr_stage2_val_report.csv`, and `reports/xgboost_stage1_lr_stage2_test_report.csv`.
   - Saves model artifact to `deploy/xgboost_stage1_lr_stage2_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)
library(xgboost)
library(nnet)
# Check for mlr3tuning + bbotk libraries
has_mlr3tuning <- requireNamespace("mlr3tuning", quietly = TRUE) && requireNamespace("bbotk", quietly = TRUE)
if (has_mlr3tuning) {
  library(mlr3)
  library(mlr3learners)
  library(mlr3tuning)
  library(bbotk)
  library(paradox)
  cat("mlr3tuning + bbotk libraries successfully loaded.\n")
} else {
  cat("Note: mlr3tuning / bbotk not installed in environment. Fallback optimization will be used.\n")
}
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 FE Inputs & Apply Controlled Class Subsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Construct 13 Clinical Feature Engineering flags
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
raw_esi <- as.character(raw_df[[target_col]])
df_feng$raw_esi <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_feng)
df_clean <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_clean), nrow(df_clean)))
# ---------------------------------------------------------
# CONTROLLED CLASS COUNT SUBSAMPLING (REGULATING DATASET COUNT PER CLASS)
# ---------------------------------------------------------
# Keep all rare ESI 1 rows and regulate majority class counts to prevent overwhelming ESI 3 bias
max_count_per_class <- 15000  # Cap majority classes to 15,000 rows max
kept_indices <- c()
for (cls in c("1", "2", "3", "4", "5")) {
  cls_idx <- which(df_clean$raw_esi == cls)
  if (length(cls_idx) > max_count_per_class) {
    cls_kept <- sample(cls_idx, size = max_count_per_class)
  } else {
    cls_kept <- cls_idx
  }
  kept_indices <- c(kept_indices, cls_kept)
}
df <- df_clean[sort(kept_indices), ]
cat(sprintf("Controlled Class Regulation Applied (Max %d rows per majority class):\n  Original: %d rows -> Regulated: %d rows\n",
            max_count_per_class, nrow(df_clean), nrow(df)))
cat("Regulated 5-Class ESI Target Distribution:\n")
print(table(df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df$raw_esi, p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$raw_esi, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Hyperparameter Optimization using mlr3tuning + bbotk
# ---------------------------------------------------------
set.seed(config$training$random_state)
best_xgb_params <- list(
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8,
  nrounds          = 150
)
if (has_mlr3tuning) {
  tryCatch({
    cat("=== Configuring mlr3tuning + bbotk Search Space ===\n")
    
    task_tune <- TaskClassif$new(id = "esi_triage", target = "raw_esi", backend = train_df)
    learner_tune <- lrn("classif.xgboost", predict_type = "prob")
    
    search_space <- ps(
      eta              = p_dbl(lower = 0.01, upper = 0.1, logscale = TRUE),
      max_depth        = p_int(lower = 3, upper = 7),
      subsample        = p_dbl(lower = 0.6, upper = 1.0),
      colsample_bytree = p_dbl(lower = 0.6, upper = 1.0),
      nrounds          = p_int(lower = 50, upper = 150)
    )
    
    resampling_tune <- rsmp("cv", folds = 3)  # Corrected rsmp syntax for mlr3
    measure_tune    <- msr("classif.logloss")
    terminator_tune <- trm("evals", n_evals = 10)
    tuner_tune      <- tnr("random_search")
    
    cat("Executing Hyperparameter Optimization via mlr3tuning + bbotk...\n")
    inst <- TuningInstanceBatchSingleCrit$new(
      task         = task_tune,
      learner      = learner_tune,
      resampling   = resampling_tune,
      measure      = measure_tune,
      search_space = search_space,
      terminator   = terminator_tune
    )
    
    tuner_tune$optimize(inst)
    res_params <- inst$result_learner_param_vals
    
    if (!is.null(res_params$eta))              best_xgb_params$eta              <- res_params$eta
    if (!is.null(res_params$max_depth))        best_xgb_params$max_depth        <- res_params$max_depth
    if (!is.null(res_params$subsample))        best_xgb_params$subsample        <- res_params$subsample
    if (!is.null(res_params$colsample_bytree)) best_xgb_params$colsample_bytree <- res_params$colsample_bytree
    if (!is.null(res_params$nrounds))          best_xgb_params$nrounds          <- res_params$nrounds
    
    cat("mlr3tuning + bbotk Optimization Complete! Best Parameters:\n")
    print(best_xgb_params)
  }, error = function(e) {
    cat("Note during mlr3tuning optimization:", e$message, "\nUsing robust default parameters.\n")
  })
} else {
  cat("Using baseline XGBoost parameters.\n")
}
cat("Final Stage 1 XGBoost Parameters:\n")
print(best_xgb_params)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Stage 1 Multi-Class XGBoost 5-Fold Stratified OOF Generation (Unweighted)
# ---------------------------------------------------------
set.seed(config$training$random_state)
feat_names <- setdiff(names(train_df), "raw_esi")
target_classes <- c("1", "2", "3", "4", "5")
num_classes    <- length(target_classes)
X_train_mat <- as.matrix(train_df[, feat_names])
y_train_num <- as.numeric(train_df$raw_esi) - 1  # 0-indexed integers for XGBoost
X_val_mat  <- as.matrix(val_df[, feat_names])
X_test_mat <- as.matrix(test_df[, feat_names])
n_folds <- 5
folds   <- createFolds(train_df$raw_esi, k = n_folds, returnTrain = FALSE)
oof_probs <- matrix(0, nrow = nrow(train_df), ncol = num_classes)
colnames(oof_probs) <- paste0("prob_P", target_classes)
val_preds_list  <- list()
test_preds_list <- list()
xgb_fold_models <- list()
xgb_params <- list(
  objective        = "multi:softprob",
  num_class        = num_classes,
  eval_metric      = "mlogloss",
  eta              = best_xgb_params$eta,
  max_depth        = best_xgb_params$max_depth,
  subsample        = best_xgb_params$subsample,
  colsample_bytree = best_xgb_params$colsample_bytree
)
cat(sprintf("Starting Stage 1 XGBoost %d-Fold Stratified CV on Regulated Data...\n", n_folds))
for (k in 1:n_folds) {
  val_idx   <- folds[[k]]
  train_idx <- setdiff(1:nrow(train_df), val_idx)
  
  dtrain_fold <- xgb.DMatrix(data = X_train_mat[train_idx, ], label = y_train_num[train_idx])
  dval_fold   <- xgb.DMatrix(data = X_train_mat[val_idx, ],   label = y_train_num[val_idx])
  
  mod_fold <- xgb.train(
    params    = xgb_params,
    data      = dtrain_fold,
    nrounds   = best_xgb_params$nrounds,
    evals     = list(train = dtrain_fold, val = dval_fold),
    early_stopping_rounds = 20,
    verbose   = 0
  )
  
  xgb_fold_models[[k]] <- mod_fold
  
  # Out-Of-Fold Predictions for fold k
  oof_pred_raw <- predict(mod_fold, newdata = dval_fold)
  oof_probs[val_idx, ] <- matrix(oof_pred_raw, ncol = num_classes, byrow = TRUE)
  
  # Stage 1 Predictions for Val and Test sets from fold k
  val_raw  <- predict(mod_fold, newdata = xgb.DMatrix(data = X_val_mat))
  test_raw <- predict(mod_fold, newdata = xgb.DMatrix(data = X_test_mat))
  
  val_preds_list[[k]]  <- matrix(val_raw,  ncol = num_classes, byrow = TRUE)
  test_preds_list[[k]] <- matrix(test_raw, ncol = num_classes, byrow = TRUE)
  
  cat(sprintf("  - Fold %d/%d complete (Best iteration: %d)\n", k, n_folds, mod_fold$best_iteration))
}
# Average predicted probabilities across all 5 folds for Validation and Test sets
val_stage1_probs  <- Reduce("+", val_preds_list) / n_folds
test_stage1_probs <- Reduce("+", test_preds_list) / n_folds
colnames(val_stage1_probs)  <- paste0("prob_P", target_classes)
colnames(test_stage1_probs) <- paste0("prob_P", target_classes)
cat("Stage 1 Multi-Class XGBoost OOF generation complete! OOF matrix shape:", dim(oof_probs)[1], "x", dim(oof_probs)[2], "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Stage 2 Unweighted Meta-Learner (Logistic Regression) Training on OOF Probabilities
# ---------------------------------------------------------
set.seed(config$training$random_state)
# Create Stage 2 Datasets using Stage 1 Probability Features
stage2_train_df <- as.data.frame(oof_probs)
stage2_train_df$raw_esi <- train_df$raw_esi
stage2_val_df <- as.data.frame(val_stage1_probs)
stage2_val_df$raw_esi <- val_df$raw_esi
stage2_test_df <- as.data.frame(test_stage1_probs)
stage2_test_df$raw_esi <- test_df$raw_esi
formula_stage2 <- as.formula("raw_esi ~ prob_P1 + prob_P2 + prob_P3 + prob_P4 + prob_P5")
cat("Training Stage 2 Meta-Learner (Multinomial Logistic Regression)...\n")
lr_stage2_model <- multinom(formula_stage2, data = stage2_train_df, trace = FALSE, MaxNWts = 5000)
cat("Stage 2 Logistic Regressor Meta-Learner Training Complete!\n")
print(summary(lr_stage2_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Comprehensive Benchmark Across All 5 ESI Classes & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_stacked_pipeline <- function(stage2_model, stage2_data, original_df, set_name) {
  prob_matrix <- predict(stage2_model, newdata = stage2_data, type = "probs")
  target_classes <- c("1", "2", "3", "4", "5")
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor   <- factor(target_classes[max_idx], levels = target_classes)
  actual_factor <- factor(original_df$raw_esi, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- cm$byClass[, "Pos Pred Value"]
  rec_by_class  <- cm$byClass[, "Sensitivity"]
  macro_prec    <- mean(prec_by_class, na.rm = TRUE)
  macro_rec     <- mean(rec_by_class,  na.rm = TRUE)
  macro_f1      <- 2 * (macro_prec * macro_rec) / (macro_prec + macro_rec + 1e-15)
  
  pr_auc_by_class <- numeric(5)
  names(pr_auc_by_class) <- target_classes
  for (cls in target_classes) {
    act_bin <- ifelse(actual_factor == cls, 1, 0)
    pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_matrix[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  roc_obj <- tryCatch(pROC::multiclass.roc(actual_factor, prob_matrix), error = function(e) NULL)
  macro_roc_auc <- if (!is.null(roc_obj)) as.numeric(roc_obj$auc) else NA
  
  actual_table <- table(actual_factor)
  pred_table   <- table(pred_factor)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  class_comparison <- data.frame(
    Class        = target_classes,
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   STAGE 1 XGBOOST -> STAGE 2 LR (REGULATED DATA) - %s SET\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro F1 Score          : %.4f\n", macro_f1))
  cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Target Class Count Comparison & Performance Summary Table:\n")
  print(class_comparison)
  
  cat("\nFull 5x5 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, macro_prec = macro_prec, macro_rec = macro_rec, macro_f1 = macro_f1, macro_pr_auc = macro_pr_auc, macro_roc_auc = macro_roc_auc, report_df = class_comparison))
}
res_val  <- evaluate_stacked_pipeline(lr_stage2_model, stage2_val_df,  val_df,  "Validation")
res_test <- evaluate_stacked_pipeline(lr_stage2_model, stage2_test_df, test_df, "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "xgboost_stage1_lr_stage2_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "xgboost_stage1_lr_stage2_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/xgboost_stage1_lr_stage2_val_report.csv\n")
cat("Test CSV Report written to:       reports/xgboost_stage1_lr_stage2_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 8: Diagnostic Plots (Metrics Summary Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split     = factor(c("Validation", "Test"), levels = c("Validation", "Test")),
  Accuracy  = c(res_val$acc,          res_test$acc),
  Precision = c(res_val$macro_prec,   res_test$macro_prec),
  Recall    = c(res_val$macro_rec,    res_test$macro_rec),
  F1_Score  = c(res_val$macro_f1,     res_test$macro_f1),
  PR_AUC    = c(res_val$macro_pr_auc, res_test$macro_pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "F1_Score", "PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3.2) +
  theme_minimal() +
  scale_fill_manual(values = c("Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Stacked Pipeline (Regulated XGBoost OOF -> Stage 2 LR): Validation vs Test",
       subtitle = "Evaluating Accuracy, Macro Precision, Recall, F1 Score, and PR-AUC",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 12), legend.position = "top")
plot_path <- file.path(plots_dir, "xgboost_stage1_lr_stage2_metrics.png")
ggsave(plot_path, plot = p_bar, width = 9.5, height = 5, dpi = 300)
cat("Metrics Comparison Plot saved to:", plot_path, "\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 9: Save Dual-Stage Pipeline Model Artifacts & Metadata
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_stage1_lr_stage2_model.rds")
saveRDS(list(
  best_xgb_params = best_xgb_params,
  xgb_fold_models = xgb_fold_models,
  lr_stage2_model = lr_stage2_model,
  preproc         = preproc,
  target_classes  = target_classes
), file = model_path)
cat("Regulated Stacked 2-Stage Model Artifact saved to:", model_path, "\n")